# FOMC Hawkish-Dovish Tone Analysis

This notebook is the commented source-code walkthrough for the CS5100 final project. It turns public Federal Reserve FOMC minutes into a hawkish-dovish policy-tone index and validates the index against Treasury yields from FRED.

## 1. Data Sources

- FOMC minutes: official public Federal Reserve meeting calendar and historical pages.
- Market data: FRED `DGS10` and `DGS2` Treasury Constant Maturity Rate CSV files.

Run the following cell only if raw data have not already been downloaded.

In [ ]:
# Optional: download official public data.
# !python3 ../scripts/fetch_data.py --start-year 2008 --end-year 2026

## 2. Sentence-Level Tone Classifier

The baseline classifier is intentionally transparent. A sentence is labeled hawkish when hawkish monetary-policy terms outnumber dovish terms, dovish when the reverse is true, and neutral otherwise. The document score is:

`net hawkish score = (hawkish sentence count - dovish sentence count) / total sentence count`

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

import pandas as pd
from IPython.display import Image, display
from analyze_tone import HAWKISH_TERMS, DOVISH_TERMS, score_sentence

print('Sample hawkish terms:', sorted(list(HAWKISH_TERMS))[:8])
print('Sample dovish terms:', sorted(list(DOVISH_TERMS))[:8])

In [ ]:
examples = [
    'Inflation pressures remained elevated and several participants supported further tightening.',
    'The Committee judged that downside risks and labor market slack warranted continued accommodation.',
    'Participants reviewed recent data on household spending and business fixed investment.'
]

for sentence in examples:
    print(score_sentence(sentence), '::', sentence)

## 3. Run the Full Analysis

The script scores every downloaded minute, aligns meeting dates with FRED yields, computes correlations, and creates figures.

In [ ]:
# Recompute scores and figures.
# !python3 ../scripts/analyze_tone.py

## 4. Processed Scores

The panel below contains one row per FOMC meeting: sentence counts, net hawkish score, smoothed score, and aligned Treasury yields.

In [ ]:
panel = pd.read_csv(PROJECT_ROOT / 'data/processed/fomc_tone_market_panel.csv')
panel.head()

In [ ]:
panel[['date', 'sentence_count', 'hawkish_sentences', 'dovish_sentences', 'net_hawkish_score', 'score_ma3', 'DGS10', 'DGS2']].tail()

## 5. Main Visualization

This chart compares the FOMC tone index with the 10-year Treasury yield and marks major policy regimes.

In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'figures/tone_vs_dgs10.png')))

## 6. Statistical Validation

The correlation table tests contemporaneous and lagged relationships between the smoothed tone score and Treasury yields. The 2-year yield should be especially sensitive to expected monetary policy.

In [ ]:
corr = pd.read_csv(PROJECT_ROOT / 'data/processed/correlation_results.csv')
corr

In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'figures/tone_yield_scatter.png')))

## 7. Optional FinBERT Extension

`scripts/finbert_optional.py` shows how to run `ProsusAI/finbert` on policy sentences. For the strongest version of the project, create 200-300 reviewed hawkish/dovish/neutral sentence labels, then fine-tune FinBERT as a three-class classifier. This avoids relying on generic positive/negative financial sentiment as a proxy for monetary-policy stance.

## 8. Conclusion

The project demonstrates an end-to-end NLP workflow with public data: acquire FOMC minutes, classify sentence-level tone, aggregate document scores, visualize policy-regime shifts, and validate the resulting index with market prices.